In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
import sys

sys.path.append('../src/')

import load_data


from load_data import load_instacart_data

In [3]:
data = load_instacart_data()

orders = data['orders']
order_products_prior = data['order_products_prior']
products = data['products']



In [5]:
products.head()

,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


In [4]:
# merge of the 2 tables to get the product names
orders_products=order_products_prior.merge(products[['product_id', 'product_name']], on='product_id', how='left')

In [5]:
# how many times each product was ordered
product_counts = orders_products['product_id'].value_counts()

# filter on top 1000 products
top_n=200
top_products = product_counts.head(top_n).index

filtered_orders_products = orders_products[orders_products['product_id'].isin(top_products)]

filtered_orders_products.head()

,order_id,product_id,add_to_cart_order,reordered,product_name
0,2,33120,1,1,Organic Egg Whites
1,2,28985,2,1,Michigan Organic Kale
5,2,17794,6,1,Carrots
9,3,33754,1,1,Total 2% with Strawberry Lowfat Greek Strained...
10,3,24838,2,1,Unsweetened Almondmilk


In [9]:
# percentage of transactions kept in scope after filtering on top_n products
percentage_transactions=len(filtered_orders_products)/len(order_products_prior)
print(percentage_transactions)

0.5400123615328116


In [6]:
# filter basket sizes to remove outliers
order_sizes=filtered_orders_products.groupby('order_id').size()

valid_orders=order_sizes[(order_sizes>3) & (order_sizes<20)].index
filtered_orders=filtered_orders_products[filtered_orders_products['order_id'].isin(valid_orders)]

In [8]:
len(filtered_orders)

14001681

In [7]:
basket_list=filtered_orders.groupby('order_id')['product_name'].apply(list).reset_index()
basket_list.columns=['order_id', 'products']

In [10]:
basket_list.head()

,order_id,products
0,2,"[Organic Egg Whites, Michigan Organic Kale, Ga..."
1,3,[Total 2% with Strawberry Lowfat Greek Straine...
2,4,"[Plain Pre-Sliced Bagels, Goldfish Cheddar Bak..."
3,5,"[Bag of Organic Bananas, Organic Raspberries, ..."
4,9,"[Organic Red Radish, Bunch, Whole White Mushro..."


In [15]:
len(basket_list)

1762585

In [16]:
sample_size = 10000
basket_sample = basket_list.sample(n=sample_size, random_state=42)

In [17]:
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()
te_ary = te.fit(basket_sample['products']).transform(basket_sample['products'])
df_encoded = pd.DataFrame(te_ary, columns=te.columns_)

frequent_itemsets = apriori(df_encoded, min_support=0.01, use_colnames=True)

In [18]:
df_encoded.head()

,0% Greek Strained Yogurt,1% Low Fat Milk,1% Lowfat Milk,100 Calorie Per Bag Popcorn,100% Apple Juice,100% Grated Parmesan Cheese,100% Lactose Free Fat Free Milk,100% Natural Spring Water,100% Pure Pumpkin,100% Raw Coconut Water,...,YoKids Squeeze! Organic Strawberry Flavor Yogurt,"YoKids Squeezers Organic Low-Fat Yogurt, Strawberry",YoKids Strawberry Banana/Strawberry Yogurt,Yobaby Organic Plain Yogurt,"Yogurt, Lowfat, Strawberry","Yogurt, Strained Low-Fat, Coconut",Yotoddler Organic Pear Spinach Mango Yogurt,Yukon Gold Potatoes 5lb Bag,ZBar Organic Chocolate Brownie Energy Snack,Zero Calorie Cola
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [19]:
# generate association rules
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.3)

print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))

rules_sorted = rules.sort_values('lift', ascending=False)
print(rules_sorted[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))

                              antecedents               consequents  support  \
0  (Organic Large Extra Fancy Fuji Apple)  (Bag of Organic Bananas)   0.0115   
1                   (Organic Raspberries)  (Bag of Organic Bananas)   0.0216   
2                        (Cucumber Kirby)                  (Banana)   0.0142   
3                      (Honeycrisp Apple)                  (Banana)   0.0135   
4                    (Organic Fuji Apple)                  (Banana)   0.0157   
5                   (Seedless Red Grapes)                  (Banana)   0.0122   
6                          (Strawberries)                  (Banana)   0.0230   
7                         (Yellow Onions)                  (Banana)   0.0116   

   confidence      lift  
0    0.347432  2.069279  
1    0.316716  1.886334  
2    0.317673  1.507705  
3    0.349741  1.659900  
4    0.368545  1.749144  
5    0.322751  1.531805  
6    0.352761  1.674232  
7    0.328612  1.559620  
                              antecedents    

Support = frequency of the association
Confidence = % of times that when I buy antecedent I get consequent
Lift = Probability of association compared to random = PB(A)/P(B)

TEST ECLAT

In [8]:
from collections import defaultdict

# Convert dataset
transactions = {}
for idx, row in basket_list.iterrows():
    transactions[f"T{idx}"] = row['products']

min_support = int(0.005 * len(transactions))

# generate tidsets
def generate_tidsets(transactions):
    item_tidset = defaultdict(set)
    for tid, items in transactions.items():
        for item in items:
            item_tidset[item].add(tid)
    return item_tidset

def eclat(prefix, items, min_support, frequent_itemsets):
    while items:
        item, tidset = items.pop()
        support = len(tidset)
        if support >= min_support:
            new_itemset = prefix + [item]
            frequent_itemsets[frozenset(new_itemset)] = support
            suffix = []
            for other_item, other_tidset in items:
                intersection = tidset & other_tidset
                if len(intersection) >= min_support:
                    suffix.append((other_item, intersection))
            if suffix:
                eclat(new_itemset, suffix, min_support, frequent_itemsets)

item_tidset = generate_tidsets(transactions)
items = sorted(item_tidset.items(), key=lambda x: len(x[1]))
frequent_itemsets = {}
eclat([], items, min_support, frequent_itemsets)

In [15]:
for itemset, support in sorted(frequent_itemsets.items(), 
                                key=lambda x: (-len(x[0]), -x[1], sorted(list(x[0])))):
    print(list(itemset), "=>", support)

['Bag of Organic Bananas', 'Organic Hass Avocado'] => 53485
['Organic Strawberries', 'Bag of Organic Bananas'] => 52236
['Organic Strawberries', 'Banana'] => 48618
['Organic Avocado', 'Banana'] => 46487
['Banana', 'Organic Baby Spinach'] => 44109
['Bag of Organic Bananas', 'Organic Baby Spinach'] => 42457
['Strawberries', 'Banana'] => 35973
['Large Lemon', 'Banana'] => 34694
['Bag of Organic Bananas', 'Organic Raspberries'] => 34430
['Organic Strawberries', 'Organic Hass Avocado'] => 33984
['Organic Strawberries', 'Organic Baby Spinach'] => 31193
['Organic Fuji Apple', 'Banana'] => 29483
['Organic Hass Avocado', 'Organic Baby Spinach'] => 29027
['Organic Strawberries', 'Organic Raspberries'] => 28185
['Banana', 'Cucumber Kirby'] => 27755
['Banana', 'Organic Whole Milk'] => 27399
['Organic Hass Avocado', 'Banana'] => 27023
['Limes', 'Banana'] => 26468
['Organic Avocado', 'Organic Baby Spinach'] => 25980
['Banana', 'Honeycrisp Apple'] => 24550
['Bag of Organic Bananas', 'Organic Whole Mi

In [9]:
# Convert in dataframe for mlxtend
eclat_df = pd.DataFrame([
    {'itemsets': itemset, 'support': support / len(transactions)} 
    for itemset, support in frequent_itemsets.items()
])
eclat_df.head()

,itemsets,support
0,(Banana),0.263650
1,"(Organic Strawberries, Banana)",0.045601
2,"(Organic Strawberries, Organic Baby Spinach, B...",0.008375
3,"(Organic Strawberries, Organic Hass Avocado, B...",0.006104
4,"(Organic Strawberries, Banana, Organic Avocado)",0.007735


In [ ]:
# Generate association rules
from mlxtend.frequent_patterns import association_rules

rules = association_rules(eclat_df, metric="confidence", min_threshold=0.3)

# Order by lift
rules_sorted = rules.sort_values('lift', ascending=False)

def extract_names(frozenset_items):
    return ', '.join(sorted(frozenset_items))

rules_sorted['antecedent'] = rules_sorted['antecedents'].apply(extract_names)
rules_sorted['consequent'] = rules_sorted['consequents'].apply(extract_names)

# Export without frozenset columns
rules_clean = rules_sorted[[
    'antecedent',
    'consequent', 
    'support',
    'confidence',
    'lift'
]]

rules_clean.to_csv('../data/processed/rules_clean.csv', index=False)

print(f"✅ {len(rules_clean)} règles exportées dans rules_clean.csv")
print(rules_clean.head(10))

✅ 95 règles exportées dans rules_clean.csv
                                           antecedent  \
94   Total 2% Lowfat Greek Strained Yogurt with Peach   
93  Total 2% Lowfat Greek Strained Yogurt With Blu...   
90  Total 2% Lowfat Greek Strained Yogurt With Blu...   
89  Total 2% with Strawberry Lowfat Greek Strained...   
91  Total 2% with Strawberry Lowfat Greek Strained...   
92   Total 2% Lowfat Greek Strained Yogurt with Peach   
88                              Sparkling Lemon Water   
87                              Sparkling Lemon Water   
86                               Lime Sparkling Water   
84                                   Bunched Cilantro   

                                           consequent   support  confidence  \
94  Total 2% Lowfat Greek Strained Yogurt With Blu...  0.005188    0.358039   
93   Total 2% Lowfat Greek Strained Yogurt with Peach  0.005188    0.342580   
90  Total 2% with Strawberry Lowfat Greek Strained...  0.007658    0.505699   
89  Total 2% 

In [14]:

# Charge ton CSV de règles
rules = pd.read_csv('../data/processed/rules_clean.csv')

# Extrait TOUS les produits uniques (antécédents + conséquents)
all_products = set()

for _, row in rules.iterrows():
    # Ajoute produits des antécédents
    all_products.update(row['antecedent'].split(', '))
    # Ajoute produits des conséquents
    all_products.update(row['consequent'].split(', '))

# Crée DataFrame
products_list = pd.DataFrame({
    'product_name': sorted(all_products)
})

# Export
products_list.to_csv('../data/processed/products_in_rules.csv', index=False)

print(f"✅ {len(products_list)} produits uniques exportés")
print(products_list.head(10))

✅ 74 produits uniques exportés
                        product_name
0             100% Raw Coconut Water
1             100% Whole Wheat Bread
2           Apple Honeycrisp Organic
3                                Bag
4             Bag of Organic Bananas
5                             Banana
6                     Bartlett Pears
7                        Blueberries
8  Boneless Skinless Chicken Breasts
9                     Broccoli Crown
